# Full modeling pipeline (baselines → fine-tunes → joint → cross-lingual → report)

Mirrors `modeling/scripts/full_modeling_pipeline.sh`: classical and zero-shot baselines per language, all transformer fine-tunes per language, joint Igbo+Yoruba model, cross-lingual Afro-XLM-R runs, then a rebuilt aggregate markdown report from `runs/`.

This can take **many hours** when smoke mode is off (Optuna on some configs, multiple full training runs). Run sections selectively if you only need part of the pipeline.

**Smoke vs full:** Edit **`kc_train/modeling/runtime_mode.py`** when using this bundle (that is the `modeling` package on `sys.path`), or set `MODELING_SMOKE=1`. Re-run the setup cell (it reloads `runtime_mode`) or restart the kernel after edits.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path


def find_kc_train_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "modeling" / "trainer.py").is_file() and (candidate / "dataset").is_dir():
            return candidate
    raise RuntimeError("cd into kc_train (folder with modeling/ and dataset/) then restart kernel.")


ROOT = find_kc_train_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("KC_TRAIN_ROOT =", ROOT)

import importlib

import modeling.runtime_mode as _runtime_mode

importlib.reload(_runtime_mode)
print("runtime_mode loaded from:", _runtime_mode.__file__)
print("SMOKE_TEST =", _runtime_mode.SMOKE_TEST)

## 1. Baselines (classical + zero-shot)

In [ ]:
from modeling.baselines import run_all_baselines

for lang in ("igbo", "yoruba"):
    print("=== baselines", lang, "===")
    run_all_baselines(lang)

## 2. Fine-tune each YAML on Igbo and Yoruba

In [ ]:
from modeling.trainer import train_from_config
from modeling.runtime_mode import SMOKE_TEST, effective_smoke, resolve_train_config_path

print("effective_smoke() =", effective_smoke(), "| SMOKE_TEST file flag =", SMOKE_TEST)
CONFIGS = [
    resolve_train_config_path(ROOT, "xlmr_base"),
    resolve_train_config_path(ROOT, "afroxlmr_base"),
    resolve_train_config_path(ROOT, "afriberta_large"),
]
for cfg in CONFIGS:
    for lang in ("igbo", "yoruba"):
        print("=== finetune", cfg.name, lang, "===")
        train_from_config(config_path=cfg, language_override=lang)

## 3. Joint multilingual fine-tune

In [ ]:
from modeling.trainer import train_from_config
from modeling.runtime_mode import resolve_train_config_path

JOINT = resolve_train_config_path(ROOT, "joint_igbo_yoruba")
train_from_config(config_path=JOINT)

## 4. Cross-lingual transfer (Afro-XLM-R)

In [ ]:
from modeling.trainer import run_cross_lingual_train_eval
from modeling.runtime_mode import resolve_train_config_path

AFRO = resolve_train_config_path(ROOT, "afroxlmr_base")
run_cross_lingual_train_eval(AFRO, "igbo", "yoruba")
run_cross_lingual_train_eval(AFRO, "yoruba", "igbo")

## 5. Rebuild aggregate markdown report

In [ ]:
from modeling.common import RUNS_DIR
from modeling.reports import rebuild_phase2_report

rebuild_phase2_report(RUNS_DIR)